# Use Globus Compute/Flows with WPS

## Setup Globus Compute Client

In [1]:
import time
import json
import datetime
import globus_sdk

from globus_sdk import TimerJob
from globus_compute_sdk import Executor
from globus_sdk.experimental.globus_app import UserApp

from globus_sdk.utils import slash_join

In [2]:
CLIENT_ID = "c781864e-a9c9-482e-8db8-d58ac5962a86"
my_app = UserApp("crocus-user-app", client_id=CLIENT_ID)

flows_client = globus_sdk.FlowsClient(app=my_app)

In [3]:
compute_endpoint = "a8ef913f-dea0-486b-b2a9-4220a9549274"

In [4]:
gce = Executor(endpoint_id=compute_endpoint)


Please authenticate with Globus here:
-------------------------------------
https://auth.globus.org/v2/oauth2/authorize?client_id=4cf29807-cf21-49ec-9443-ff9a3fb9f81c&redirect_uri=https%3A%2F%2Fauth.globus.org%2Fv2%2Fweb%2Fauth-code&scope=https%3A%2F%2Fauth.globus.org%2Fscopes%2Ffacd7ccc-c5f4-42aa-916b-a0e270e2c2a9%2Fall+openid&state=_default&response_type=code&code_challenge=oFHm4r811L6ip3qh5mJROVm4pSnSc3OBnalEGV3iqIE&code_challenge_method=S256&access_type=online&prefill_named_grant=evswl134.evs.anl.gov+on+evswl134.evs.anl.gov
-------------------------------------




KeyboardInterrupt



In [16]:
def average_subset_by_time(node="DKRZ",
                           start_date="1990-01-01",
                           end_date="2000-01-01",
                           lat_min=0,
                           lat_max=35,
                           lon_min=65,
                           lon_max=100,
                           average_frequency="year",
                           experiment_id=["historical"],
                           variable_id=["tas"],
                           member_id=["r1i1p1f1"],
                           table_id=["Amon"],
                           institution_id=["MIROC"]
                           ):
    """
    Parameters
    ==========
    node: str, where you would like to run the WPS
        options: DKRZ, ORNL, ANL
    
    variable: str
        options: must be a valid variable in the vocabulary
    
    start_date: str
        Start date in YYYY-MM-DD for the temporal subset
    
    end_date: str
        End date in YYYY-MM-DD for the temporal subset
        
    lat_min: int
        Minimum latitude for the spatial subset
    
    lat_max: int
        Maximum latitude for the spatial subset
        
    lon_min: int
        Minimum longitude for the spatial subset
    
    lon_max: int
        Maximum longitude for the spatial subset
    
    average_frequency: str
        options: "year", "month", "day"
    
    """
    import os
    
    from rooki import operators as ops
    from rooki import rooki
    import intake_esgf
    from intake_esgf import ESGFCatalog
    
    if node == "ORNL":
        intake_esgf.conf.set(indices={"anl-dev": False,
                                      "ornl-dev": True})
        
        def build_rooki_id(id_list):
            rooki_id = id_list[0]
            rooki_id = rooki_id.split("|")[0]
            rooki_id = f"css03_data.{rooki_id}"  # <-- just something you have to know for now :(
            return rooki_id
        

    elif node == "DKRZ":
        intake_esgf.conf.set(indices={"anl-dev": False,
                                      "ornl-dev": False,
                                      "esgf-node.llnl.gov": True})
        
        def build_rooki_id(id_list):
            rooki_id = id_list[0]
            rooki_id = rooki_id.split("|")[0]
            return rooki_id
        
    else:
        raise NameError("Node not in allowed list ['ORNL', 'DKRZ']")
        
    
    def run_workflow(variable_id, rooki_id):
        workflow = ops.AverageByTime(
            ops.Subset(
            ops.Input(variable_id, [rooki_id]),
            time=f"{start_date}/{end_date}",
            area=f"{lon_min},{lat_min},{lon_max},{lat_max}",
        ),
        freq=average_frequency,
        )
    
        response = workflow.orchestrate()
        if not response.ok:
            raise ValueError(response)
        return response.download()[0]
        
    
    # Search the catalog
    cat = ESGFCatalog().search(experiment_id=experiment_id,
                               variable_id=variable_id,
                               member_id=member_id,
                               table_id=table_id,
                               institution_id=institution_id
                              )
    
    # Apply the id building
    rooki_ids = cat.df.id.apply(build_rooki_id)
    
    
    dset_dict = {}
    for rooki_id in rooki_ids:
        dset_dict[rooki_id] = run_workflow(variable_id[0], rooki_id)
    
    return dset_dict

In [17]:
average_subset_by_time(variable_id=['tas', 'uas', 'vas'])

   Searching indices:   0%|          |0/1 [       ?index/s]

{'CMIP6.CMIP.MIROC.MIROC6.historical.r1i1p1f1.Amon.vas.gn.v20181212': '/var/folders/bw/c9j8z20x45s2y20vv6528qjc0000gq/T/metalink_x2fx8dkw/vas_Amon_MIROC6_historical_r1i1p1f1_gn_19900101-19990101_avg-year.nc',
 'CMIP6.CMIP.MIROC.MIROC6.historical.r1i1p1f1.Amon.uas.gn.v20181212': '/var/folders/bw/c9j8z20x45s2y20vv6528qjc0000gq/T/metalink_fgyk82bg/uas_Amon_MIROC6_historical_r1i1p1f1_gn_19900101-19990101_avg-year.nc',
 'CMIP6.CMIP.MIROC.MIROC6.historical.r1i1p1f1.Amon.tas.gn.v20181212': '/var/folders/bw/c9j8z20x45s2y20vv6528qjc0000gq/T/metalink_6csjvnxc/tas_Amon_MIROC6_historical_r1i1p1f1_gn_19900101-19990101_avg-year.nc'}